# Modeling: churn prediction with four classifiers

This notebook trains Logistic Regression, Random Forest, XGBoost, and LightGBM
on the telecom churn dataset. We use GridSearchCV for hyperparameter tuning,
handle class imbalance, and cross-validate using ROC-AUC.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '..')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_curve
)

from src.data_loader import load_and_prepare
from src.model import _get_models_and_grids, RANDOM_STATE

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Stratified train/test split

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_prepare(
    filepath='../data/telco_churn.csv'
)

print(f'\nClass distribution (train): {np.bincount(y_train)}')
print(f'Class distribution (test):  {np.bincount(y_test)}')
print(f'Imbalance ratio: {(y_train == 0).sum() / (y_train == 1).sum():.2f} : 1')

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 2. Handle class imbalance

The churn rate is approximately 26%, so the classes are moderately imbalanced.
We address this through:
- `class_weight='balanced'` for Logistic Regression and Random Forest
- `scale_pos_weight` for XGBoost (ratio of negative to positive samples)
- `is_unbalance=True` for LightGBM
- Stratified cross-validation to maintain class proportions in each fold

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight for XGBoost: {scale_pos_weight:.2f}')

## 3. Define models and hyperparameter grids

In [ ]:
models_config = {
    'Logistic Regression': {
        'model': LogisticRegression(
            random_state=42, max_iter=1000, class_weight='balanced'
        ),
        'params': {'C': [0.01, 0.1, 1.0, 10.0]},
        'needs_scaling': True,
    },
    'Random Forest': {
        'model': RandomForestClassifier(
            random_state=42, n_jobs=-1, class_weight='balanced'
        ),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [5, 10, 15],
            'min_samples_split': [2, 5],
        },
        'needs_scaling': False,
    },
    'XGBoost': {
        'model': XGBClassifier(
            random_state=42, eval_metric='logloss',
            use_label_encoder=False, verbosity=0,
            scale_pos_weight=scale_pos_weight,
        ),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.05, 0.1],
        },
        'needs_scaling': False,
    },
    'LightGBM': {
        'model': LGBMClassifier(
            random_state=42, verbose=-1, n_jobs=-1, is_unbalance=True
        ),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7, -1],
            'learning_rate': [0.05, 0.1],
        },
        'needs_scaling': False,
    },
}

for name, cfg in models_config.items():
    n_combos = 1
    for vals in cfg['params'].values():
        n_combos *= len(vals)
    print(f'{name}: {n_combos} parameter combinations')

## 4. GridSearchCV training with ROC-AUC

In [ ]:
results = {}
trained_models = {}

for name, config in models_config.items():
    print(f'\nTraining {name}...')
    Xtr = X_train_scaled if config['needs_scaling'] else X_train
    Xte = X_test_scaled if config['needs_scaling'] else X_test

    grid = GridSearchCV(
        config['model'], config['params'],
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1, refit=True,
    )
    grid.fit(Xtr, y_train)

    best_model = grid.best_estimator_
    trained_models[name] = {'model': best_model, 'needs_scaling': config['needs_scaling']}

    y_pred = best_model.predict(Xte)
    y_prob = best_model.predict_proba(Xte)[:, 1]

    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc_roc': roc_auc_score(y_test, y_prob),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'y_prob': y_prob,
        'best_params': grid.best_params_,
    }

    print(f'  Best params: {grid.best_params_}')
    print(f'  AUC-ROC: {results[name]["auc_roc"]:.4f}')
    print(f'  F1:      {results[name]["f1"]:.4f}')

## 5. Cross-validation with ROC-AUC

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {}

for name, info in trained_models.items():
    Xtr = X_train_scaled if info['needs_scaling'] else X_train
    scores = cross_val_score(
        info['model'], Xtr, y_train, cv=cv, scoring='roc_auc', n_jobs=-1
    )
    cv_scores[name] = scores
    print(f'{name:25s}  CV AUC: {scores.mean():.4f} (+/- {scores.std():.4f})')

# Boxplot
fig, ax = plt.subplots(figsize=(8, 4))
bp = ax.boxplot([cv_scores[n] for n in cv_scores],
                labels=list(cv_scores.keys()), patch_artist=True)
colors = ['#90CAF9', '#4CAF50', '#FF9800', '#9C27B0']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_ylabel('ROC-AUC')
ax.set_title('5-fold stratified CV comparison')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Model comparison table

In [ ]:
comparison = pd.DataFrame({
    name: {k: v for k, v in r.items()
           if k not in ('confusion_matrix', 'y_prob', 'best_params')}
    for name, r in results.items()
}).T.round(4)

comparison = comparison.sort_values('auc_roc', ascending=False)
print('Model comparison (sorted by AUC-ROC):')
comparison

In [ ]:
# Grouped bar chart
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
x = np.arange(len(metrics_to_plot))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#90CAF9', '#4CAF50', '#FF9800', '#9C27B0']

for i, (name, row) in enumerate(comparison.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    ax.bar(x + i * width, values, width, label=name, color=colors[i])

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel('Score')
ax.set_title('Model comparison across metrics')
ax.legend(loc='lower right')
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. Best model selection

In [ ]:
best_name = comparison.index[0]  # already sorted by AUC-ROC
best_auc = results[best_name]['auc_roc']
best_params = results[best_name]['best_params']

print(f'Best model: {best_name}')
print(f'AUC-ROC:    {best_auc:.4f}')
print(f'Parameters: {best_params}')
print(f'\nConfusion matrix:')
cm = results[best_name]['confusion_matrix']
print(f'  TN={cm[0,0]:4d}  FP={cm[0,1]:4d}')
print(f'  FN={cm[1,0]:4d}  TP={cm[1,1]:4d}')

## Summary

All four models were tuned with GridSearchCV using stratified 5-fold CV
and ROC-AUC scoring. Key findings:

- Class imbalance was handled through class weights and scale_pos_weight
- XGBoost and LightGBM typically achieve the highest AUC-ROC
- Logistic Regression provides a solid baseline with interpretable coefficients
- Random Forest offers a middle ground between interpretability and performance

Next: detailed evaluation with ROC curves, SHAP analysis, and business
impact estimation in `04_evaluation.ipynb`.